In [1]:
import sys
!{sys.executable} -m pip install chromadb==1.5.9 sentence-transformers==5.7.0 pandas numpy ollama==0.6.2 python-dotenv

'C:\Users\Abdelrahman' is not recognized as an internal or external command,
operable program or batch file.


In [2]:
import os
import glob

DOCS_DIR = '../docs/sample_docs'

docs = []
for path in glob.glob(os.path.join(DOCS_DIR, '*.txt')):
    with open(path, encoding='utf-8') as f:
        text = f.read()
    filename = os.path.basename(path)
    docs.append({'filename': filename, 'text': text, 'char_count': len(text)})
    print(f'{filename}: {len(text)} chars, {text.count(chr(10))} lines')

print(f'\nTotal documents: {len(docs)}')

fastapi_guide.txt: 2583 chars, 22 lines
python_advanced.txt: 2718 chars, 22 lines
python_basics.txt: 2891 chars, 25 lines
software_engineering.txt: 2454 chars, 19 lines

Total documents: 4


In [3]:
import re

CHUNK_SIZE = 500   
OVERLAP    = 50     


def chunk_by_qa(text: str, filename: str) -> list[dict]:
    """Split on Q: boundaries first, then fixed-size if a chunk is too long."""
    # Split on every 'Q:' that starts a line
    raw_sections = re.split(r'(?=\nQ:)', text)
    chunks = []
    for i, section in enumerate(raw_sections):
        section = section.strip()
        if not section:
            continue
        if len(section) <= CHUNK_SIZE:
            chunks.append({'text': section, 'source': filename,
                           'chunk_id': f'{filename}_{i}'})
        else:
            # fallback: fixed-size with overlap
            start = 0
            sub = 0
            while start < len(section):
                end = start + CHUNK_SIZE
                chunks.append({'text': section[start:end], 'source': filename,
                               'chunk_id': f'{filename}_{i}_{sub}'})
                start = end - OVERLAP
                sub += 1
    return chunks


all_chunks = []
for doc in docs:
    doc_chunks = chunk_by_qa(doc['text'], doc['filename'])
    all_chunks.extend(doc_chunks)
    print(f"{doc['filename']}: {len(doc_chunks)} chunks")

print(f'\nTotal chunks: {len(all_chunks)}')
print('\nExample chunk:')
print(all_chunks[2]['text'][:300])

fastapi_guide.txt: 8 chunks
python_advanced.txt: 8 chunks
python_basics.txt: 9 chunks
software_engineering.txt: 7 chunks

Total chunks: 32

Example chunk:
Q: How do you define a Pydantic model in FastAPI?
Define models by inheriting from BaseModel: from pydantic import BaseModel; class Item(BaseModel): name: str; price: float; in_stock: bool = True. FastAPI uses these for request body validation, response serialization, and automatic docs. Pydantic va


In [4]:
import chromadb
from sentence_transformers import SentenceTransformer

VECTOR_STORE_PATH = '../backend/data/vector_store'
COLLECTION_NAME   = 'rag_docs'
EMBEDDING_MODEL   = 'all-MiniLM-L6-v2'

print('Loading embedding model...')
embedder = SentenceTransformer(EMBEDDING_MODEL)

client = chromadb.PersistentClient(path=VECTOR_STORE_PATH)


try:
    client.delete_collection(COLLECTION_NAME)
except Exception:
    pass
collection = client.create_collection(COLLECTION_NAME)

texts     = [c['text']     for c in all_chunks]
ids       = [c['chunk_id'] for c in all_chunks]
metadatas = [{'source': c['source'], 'chunk_id': c['chunk_id']} for c in all_chunks]

print(f'Embedding {len(texts)} chunks...')
embeddings = embedder.encode(texts, show_progress_bar=True).tolist()

collection.add(documents=texts, embeddings=embeddings, ids=ids, metadatas=metadatas)

print(f'\nStored {collection.count()} chunks in ChromaDB at: {VECTOR_STORE_PATH}')

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\Abdelrahman zayed\Downloads\rag-assistant\backend\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Abdelrahman zayed\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding 32 chunks...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Stored 32 chunks in ChromaDB at: ../backend/data/vector_store


In [5]:
def retrieve(question: str, top_k: int = 3) -> list[dict]:
    embedding = embedder.encode(question).tolist()
    results = collection.query(
        query_embeddings=[embedding],
        n_results=top_k,
        include=['documents', 'metadatas'],
    )
    return [
        {'text': doc, 'source': meta['source'], 'chunk_id': meta['chunk_id']}
        for doc, meta in zip(results['documents'][0], results['metadatas'][0])
    ]


test_questions = [
    'What is a Python generator?',
    'How do I create a virtual environment?',
    'What is the difference between list and tuple?',
    'What is the GIL in Python?',
    'How does async/await work?',
    'What is SOLID design?',
    'How do I add CORS in FastAPI?',
    'What is the FastAPI lifespan event?',
    'What is TDD?',
    'What are Python dataclasses?',
]

for q in test_questions:
    chunks = retrieve(q, top_k=1)
    print(f'Q: {q}')
    print(f'  → [{chunks[0]["source"]}] {chunks[0]["text"][:100]}...')
    print()

Q: What is a Python generator?
  → [python_advanced.txt] Q: What is a generator in Python?
A generator is a function that returns an iterator using the yield...

Q: How do I create a virtual environment?
  → [python_basics.txt] Q: What are Python virtual environments and why use them?
A virtual environment is an isolated Pytho...

Q: What is the difference between list and tuple?
  → [python_basics.txt] Q: What is the difference between list and tuple in Python?
Lists are mutable (can be changed after ...

Q: What is the GIL in Python?
  → [python_advanced.txt] Q: What is the Global Interpreter Lock (GIL)?
The GIL is a mutex in CPython that allows only one thr...

Q: How does async/await work?
  → [python_advanced.txt] Q: How do async/await work in Python?
Async/await enables asynchronous programming using coroutines....

Q: What is SOLID design?
  → [software_engineering.txt] Q: What is SOLID in software design?
SOLID is five object-oriented design principles: Single Respons...

Q: Ho

In [6]:
import ollama

OLLAMA_MODEL = 'llama3.2'

SYSTEM_PROMPT = """You are a helpful document assistant.
Answer ONLY using the provided context chunks.
Cite which chunk you used at the end of your answer.
If the context is insufficient, say so."""


def rag_answer(question: str, top_k: int = 3) -> dict:
    chunks = retrieve(question, top_k=top_k)
    context = '\n\n---\n\n'.join(
        f'[Source: {c["source"]} | Chunk: {c["chunk_id"]}]\n{c["text"]}'
        for c in chunks
    )
    prompt = f'Context:\n{context}\n\nQuestion: {question}'
    resp = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': prompt},
        ],
        options={'temperature': 0.1},
    )
    return {'answer': resp['message']['content'], 'sources': chunks}


try:
    result = rag_answer('What is a Python generator?')
    print('Answer:', result['answer'])
    print('Source:', result['sources'][0]['source'])
except Exception as e:
    print(f'Ollama not available: {e}')
    print('Start Ollama with: ollama serve && ollama pull llama3.2')

Answer: A Python generator is a function that returns an iterator using the yield keyword. It produces values one at a time, making it memory-efficient for large datasets.

[Used: python_advanced.txt_1]
Source: python_advanced.txt


In [9]:
import pandas as pd

eval_set = [
    {'question': 'What is a Python generator?',
     'expected_keyword': 'yield', 'expected_source': 'python_advanced.txt'},
    {'question': 'How do I create a virtual environment?',
     'expected_keyword': 'venv', 'expected_source': 'python_basics.txt'},
    {'question': 'What is the difference between list and tuple?',
     'expected_keyword': 'mutable', 'expected_source': 'python_basics.txt'},
    {'question': 'What is the GIL?',
     'expected_keyword': 'mutex', 'expected_source': 'python_advanced.txt'},
    {'question': 'How does async/await work in Python?',
     'expected_keyword': 'coroutine', 'expected_source': 'python_advanced.txt'},
    {'question': 'What are the SOLID principles?',
     'expected_keyword': 'Single Responsibility', 'expected_source': 'software_engineering.txt'},
    {'question': 'How do I add CORS middleware to FastAPI?',
     'expected_keyword': 'CORSMiddleware', 'expected_source': 'fastapi_guide.txt'},
    {'question': 'What is the FastAPI lifespan event?',
     'expected_keyword': 'startup', 'expected_source': 'fastapi_guide.txt'},
    {'question': 'What is Test-Driven Development?',
     'expected_keyword': 'Red', 'expected_source': 'software_engineering.txt'},
    {'question': 'What are Python dataclasses?',
     'expected_keyword': 'dataclass', 'expected_source': 'python_advanced.txt'},
]

rows = []
for item in eval_set:
    chunks = retrieve(item['question'], top_k=1)
    top_chunk = chunks[0]
    keyword_found = item['expected_keyword'].lower() in top_chunk['text'].lower()
    source_match  = item['expected_source'] in top_chunk['source']
    correct = keyword_found and source_match
    rows.append({
        'Question': item['question'][:55],
        'Retrieved Source': top_chunk['source'],
        'Keyword Found': '✅' if keyword_found else '❌',
        'Source Match': '✅' if source_match else '❌',
        'Correct': '✅' if correct else '❌',
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))
accuracy = sum(1 for r in rows if r['Correct'] == '✅') / len(rows)
print(f'\nRetrieval accuracy: {accuracy:.0%}')

                                      Question         Retrieved Source Keyword Found Source Match Correct
                   What is a Python generator?      python_advanced.txt             ✅            ✅       ✅
        How do I create a virtual environment?        python_basics.txt             ✅            ✅       ✅
What is the difference between list and tuple?        python_basics.txt             ✅            ✅       ✅
                              What is the GIL?      python_advanced.txt             ✅            ✅       ✅
          How does async/await work in Python?      python_advanced.txt             ✅            ✅       ✅
                What are the SOLID principles? software_engineering.txt             ✅            ✅       ✅
      How do I add CORS middleware to FastAPI?        fastapi_guide.txt             ✅            ✅       ✅
           What is the FastAPI lifespan event?        fastapi_guide.txt             ✅            ✅       ✅
              What is Test-Driven Dev

In [10]:
import json

config = {
    'embedding_model': EMBEDDING_MODEL,
    'collection_name': COLLECTION_NAME,
    'chunk_size': CHUNK_SIZE,
    'overlap': OVERLAP,
    'total_chunks': collection.count(),
    'documents': [d['filename'] for d in docs],
}

config_path = '../backend/data/vector_store_config.json'
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print('Exported config to:', config_path)
print(json.dumps(config, indent=2))

Exported config to: ../backend/data/vector_store_config.json
{
  "embedding_model": "all-MiniLM-L6-v2",
  "collection_name": "rag_docs",
  "chunk_size": 500,
  "overlap": 50,
  "total_chunks": 32,
  "documents": [
    "fastapi_guide.txt",
    "python_advanced.txt",
    "python_basics.txt",
    "software_engineering.txt"
  ]
}
